# Milestone 1: NLP Foundation & Semantic Similarity

## Q1: Frequency Distribution

In [ ]:
import pandas as pd
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
print(train['answer'].value_counts())
# Sum of most and least frequent: 814

## Q2: Text Cleaning & Vocabulary Size

In [ ]:
import string
def clean_and_split(text):
    text = str(text).lower()
    for p in string.punctuation:
        text = text.replace(p, '')
    return set(text.split())

vocab = set()
for prompt in train['prompt']:
    vocab.update(clean_and_split(prompt))
print("Vocabulary size:", len(vocab)) # 859

## Q3: Stop Words Filtering

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
words = clean_and_split(train.loc[1, 'prompt'])
filtered = [w for w in words if w not in ENGLISH_STOP_WORDS]
print("Words left:", len(filtered)) # 13

## Q4: TF-IDF Vectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
all_texts = train['prompt'].tolist()
for col in ['A', 'B', 'C', 'D', 'E']:
    all_texts.extend(train[col].astype(str).tolist())
    
tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(all_texts)
print("Vocab size:", len(tfidf.vocabulary_)) # 2762

## Q5 & Q6: Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
# Q5
p_vec = tfidf.transform([str(train.loc[1, 'prompt'])])
a_vec = tfidf.transform([str(train.loc[1, 'A'])])
print("Sim:", cosine_similarity(p_vec, a_vec)[0][0]) # 0.272

# Q6
correct_highest = 0
for i, row in train.iterrows():
    p_vec = tfidf.transform([str(row['prompt'])])
    sims = []
    for l in ['A', 'B', 'C', 'D', 'E']:
        sims.append((l, cosine_similarity(p_vec, tfidf.transform([str(row[l])]))[0][0]))
    best = max(sims, key=lambda x: x[1])[0]
    if best == row['answer']:
        correct_highest += 1
print("Percentage:", (correct_highest / len(train)) * 100) # 13.55%

## Q7 & Q8 & Q9 & Q10: MAP@3 and Baselines

In [ ]:
def map_at_3(y_true, y_pred_list):
    scores = []
    for t, p in zip(y_true, y_pred_list):
        score = 0
        for i, pred in enumerate(p[:3]):
            if t == pred:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return sum(scores) / len(scores)

# Q9 Majority Class
most_freq = train['answer'].mode()[0]
other_freq = train['answer'].value_counts().index.tolist()
pred = [other_freq[:3] for _ in range(len(train))]
print("Majority MAP@3:", map_at_3(train['answer'].tolist(), pred)) # 0.4212

# Q10 TF-IDF Pipeline MAP@3
tfidf_preds = []
for i, row in train.iterrows():
    p_vec = tfidf.transform([str(row['prompt'])])
    sims = []
    for l in ['A', 'B', 'C', 'D', 'E']:
        sims.append((l, cosine_similarity(p_vec, tfidf.transform([str(row[l])]))[0][0]))
    sims.sort(key=lambda x: x[1], reverse=True)
    tfidf_preds.append([x[0] for x in sims[:3]])
print("TF-IDF MAP@3:", map_at_3(train['answer'].tolist(), tfidf_preds)) # 0.29616